# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os
from getpass import getpass
os.environ["HF_TOKEN"] = getpass("Paste your Hugging Face token: ")

In [2]:
import duckdb
con = duckdb.connect()
con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")
con.execute(f"""
    CREATE SECRET hf_token (
        TYPE HUGGINGFACE,
        TOKEN '{os.environ["HF_TOKEN"]}'
    );
""")
table_path = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"

In [3]:
import pandas as pd

df = con.execute(f"""
    SELECT 
        content_hash_id, client_hash_id,
        gsc_avg_position, gsc_impressions, gsc_clicks,
        gsc_clicks * 1.0 / NULLIF(gsc_impressions, 0) as actual_ctr
    FROM read_parquet('{table_path}')
    WHERE gsc_data_available IS TRUE 
      AND gsc_impressions > 0
      AND gsc_avg_position <= 10
""").df()

df.shape

(2183484, 6)

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

**Method:** Logistic Regression, moving to Random Forest if it helps.

My lane is a ranking problem ("which pages should be reviewed first?"), so per the 
method guide, I start with a classifier's predicted probability, evaluated at 
precision@K — not a plain score. Logistic Regression is a good starting point 
because it is simple and readable: I can see exactly which features push a page's 
probability up or down. If Random Forest clearly beats it on the same metric, I will 
report that instead, but I will not add complexity unless it earns its place.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

**Split design:** Grouped by client, not random and not time-based. I split clients 
themselves into a train group and a test group, so all of a client's pages end up 
entirely in one side or the other. This is honest for my question because pages from 
the same client tend to be similar (same site structure, same content style), so a 
random row-level split could let the model see pages from a client it's "supposed" to 
be tested on, making the test score misleadingly good. A time-based split isn't 
necessary here since I'm using a single month, not predicting future dates.

In [4]:
from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(df, groups=df["client_hash_id"]))

train_df = df.iloc[train_idx]
test_df = df.iloc[test_idx]

print("Train rows:", len(train_df), "| Train clients:", train_df["client_hash_id"].nunique())
print("Test rows:", len(test_df), "| Test clients:", test_df["client_hash_id"].nunique())

Train rows: 1566537 | Train clients: 37
Test rows: 616947 | Test clients: 10


In [7]:
import numpy as np
import numpy as np

train_df = train_df.copy()
test_df = test_df.copy()

train_df["position_bucket"] = pd.cut(train_df["gsc_avg_position"], bins=[0, 3, 6, 10])
test_df["position_bucket"] = pd.cut(test_df["gsc_avg_position"], bins=[0, 3, 6, 10])

bucket_avg = train_df.groupby("position_bucket")["actual_ctr"].mean()

train_df["expected_ctr"] = train_df["position_bucket"].map(bucket_avg).astype(float)
test_df["expected_ctr"] = test_df["position_bucket"].map(bucket_avg).astype(float)

train_df["ctr_gap"] = train_df["expected_ctr"] - train_df["actual_ctr"]
test_df["ctr_gap"] = test_df["expected_ctr"] - test_df["actual_ctr"]

gap_threshold = train_df["ctr_gap"].quantile(0.8)

train_df["is_opportunity"] = ((train_df["ctr_gap"] >= gap_threshold) & (train_df["gsc_impressions"] >= 100)).astype(int)
test_df["is_opportunity"] = ((test_df["ctr_gap"] >= gap_threshold) & (test_df["gsc_impressions"] >= 100)).astype(int)

print("Train opportunity rate:", train_df["is_opportunity"].mean())
print("Test opportunity rate:", test_df["is_opportunity"].mean())

Train opportunity rate: 0.025015687468601125
Test opportunity rate: 0.04094354944590054


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [9]:
from sklearn.linear_model import LogisticRegression

features = ["gsc_avg_position", "gsc_impressions", "ctr_gap"]

# drop any rows with missing values before training
train_df = train_df.dropna(subset=features + ["is_opportunity"])
test_df = test_df.dropna(subset=features + ["is_opportunity"])

X_train = train_df[features]
y_train = train_df["is_opportunity"]

X_test = test_df[features]
y_test = test_df["is_opportunity"]

model = LogisticRegression(random_state=42, max_iter=1000)
model.fit(X_train, y_train)

test_df["model_score"] = model.predict_proba(X_test)[:, 1]

print("Train rows after dropna:", len(train_df))
print("Test rows after dropna:", len(test_df))

Train rows after dropna: 1452977
Test rows after dropna: 567318


In [10]:
# baseline score: recreate your Week 4 rule on the test set
test_df["baseline_score"] = test_df["ctr_gap"] * test_df["gsc_impressions"]

# sort by baseline score, take top 50, check how many were real opportunities
top50_baseline = test_df.sort_values("baseline_score", ascending=False).head(50)
baseline_precision = top50_baseline["is_opportunity"].mean()

# sort by model score, take top 50, check how many were real opportunities
top50_model = test_df.sort_values("model_score", ascending=False).head(50)
model_precision = top50_model["is_opportunity"].mean()

# base rate: what we'd get by guessing randomly
base_rate = test_df["is_opportunity"].mean()

print("Baseline precision@50:", baseline_precision)
print("Model precision@50:", model_precision)
print("Base rate:", base_rate)

Baseline precision@50: 0.36
Model precision@50: 0.3
Base rate: 0.044525292692987


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [11]:
importance = pd.DataFrame({
    "feature": features,
    "coefficient": model.coef_[0]
})
importance

,feature,coefficient
0,gsc_avg_position,-0.744188
1,gsc_impressions,0.001063
2,ctr_gap,42.326313


In [13]:
# false positives: model said "opportunity" but it wasn't
false_positives = test_df[(test_df["model_score"] > 0.5) & (test_df["is_opportunity"] == 0)]
false_positives[features + ["is_opportunity", "model_score"]].head(3)


,gsc_avg_position,gsc_impressions,ctr_gap,is_opportunity,model_score
829,3.476852,6912,0.001246,0,0.964305
5306,1.938975,5932,-0.005961,0,0.956658
29052,1.621578,8036,0.005209,0,0.997623


In [14]:
# false negatives: model missed a real opportunity
false_negatives = test_df[(test_df["model_score"] < 0.5) & (test_df["is_opportunity"] == 1)]
false_negatives[features + ["is_opportunity", "model_score"]].head(3)

,gsc_avg_position,gsc_impressions,ctr_gap,is_opportunity,model_score
32,2.082090,134,0.005333,1,0.063216
195,0.802260,531,0.005333,1,0.210559
201,1.924157,356,0.005333,1,0.087668


The model's coefficients show it leans almost entirely on `ctr_gap` (coefficient 42.3), 
with `gsc_avg_position` having a smaller effect and `gsc_impressions` looking nearly 
zero in the coefficient table. But looking at real wrong cases tells a different story: 
false positives all have very high impressions (6,000-8,000) but a tiny ctr_gap, and 
the model still scored them 0.96+ as opportunities. False negatives all have low 
impressions (under 600) despite having a real, qualifying gap, and the model scored 
them low (under 0.25). This means the model is more sensitive to raw impression volume 
than the coefficient table alone suggests — likely because impressions can be in the 
thousands, so even a small coefficient has a large effect once multiplied by a big 
number. My baseline rule handles this more directly by using gap × impressions 
together as one combined score, which may explain why it slightly outperformed the 
model at precision@50 (36% vs 30%). This is a fair result to report honestly, not a 
failure — a simple, well-designed rule can beat a first-pass model, especially when 
the label itself was built from the same logic the rule already uses.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.